# Cell 1. 라이브러리 import 및 경로 설정

# 환경

In [1]:
import os
import re
import ast
import json
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image

from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
    BitsAndBytesConfig
)

print("라이브러리 import 완료")

라이브러리 import 완료


In [2]:
# 현재 작업 위치 확인
print("현재 작업 위치:", os.getcwd())

# 프레임 이미지들이 들어있는 폴더
FRAMES_ROOT = Path("data/frames_for_vlm")

# 결과 저장 폴더
RESULT_DIR = Path("results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# 결과 저장 파일 경로
CSV_PATH = RESULT_DIR / "qwen_vlm_1fps_results.csv"
JSON_PATH = RESULT_DIR / "qwen_vlm_1fps_results.json"

# 모델 ID
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

# 이미지 입력 크기
IMAGE_SIZE = 224

# 생성 토큰 수
MAX_NEW_TOKENS = 512

print("FRAMES_ROOT 존재 여부:", FRAMES_ROOT.exists())
print("FRAMES_ROOT 절대 경로:", FRAMES_ROOT.resolve())
print("RESULT_DIR:", RESULT_DIR.resolve())

현재 작업 위치: c:\Users\user\Desktop\Uk_folder\sparta_bootcamp\project_collection\final_project\final_project\works\Hyeong_Uk\VLM_test
FRAMES_ROOT 존재 여부: True
FRAMES_ROOT 절대 경로: C:\Users\user\Desktop\Uk_folder\sparta_bootcamp\project_collection\final_project\final_project\works\Hyeong_Uk\VLM_test\data\frames_for_vlm
RESULT_DIR: C:\Users\user\Desktop\Uk_folder\sparta_bootcamp\project_collection\final_project\final_project\works\Hyeong_Uk\VLM_test\results


# Cell 2. Qwen2.5-VL 모델 로드

In [3]:
torch.cuda.empty_cache()

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map={"": 0},
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model.eval()

print("Qwen2.5-VL 모델 로드 완료")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Qwen2.5-VL 모델 로드 완료


# Cell 3. video_id 폴더 목록 확인

In [4]:
video_dirs = sorted([
    p for p in FRAMES_ROOT.iterdir()
    if p.is_dir()
])

print("영상 폴더 개수:", len(video_dirs))

for p in video_dirs:
    print("-", p.name)

영상 폴더 개수: 8
- knCQBlBKSRM
- LRmLOsmMHts
- mfi0n3oNABM
- MIJR3Dm0YOk
- NCVxpXxTMSU
- twi9zUxsXu0
- Wkiz8JVPvXA
- Xfu1kCID0Ls


# Cell 4. 프레임 이미지 수집 함수

In [5]:
def gather_images(video_folder: Path):
    """
    특정 video_id 폴더 안의 프레임 이미지 경로를 정렬해서 가져오는 함수
    """
    image_paths = sorted([
        p for p in video_folder.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ])
    
    return image_paths

# Cell 5. 영상별 프레임 수 확인

In [6]:
for video_dir in video_dirs:
    image_paths = gather_images(video_dir)
    print(video_dir.name, "프레임 수:", len(image_paths))

knCQBlBKSRM 프레임 수: 21
LRmLOsmMHts 프레임 수: 30
mfi0n3oNABM 프레임 수: 56
MIJR3Dm0YOk 프레임 수: 50
NCVxpXxTMSU 프레임 수: 12
twi9zUxsXu0 프레임 수: 55
Wkiz8JVPvXA 프레임 수: 30
Xfu1kCID0Ls 프레임 수: 23


# Cell 6. 이미지 로드 함수

In [7]:
def load_images(image_paths, image_size=224):
    """
    이미지 경로 리스트를 PIL Image 리스트로 변환
    Qwen VLM 입력용으로 RGB 변환 + 224x224 리사이즈
    """
    images = []

    for path in image_paths:
        img = Image.open(path).convert("RGB").resize((image_size, image_size))
        images.append(img)

    return images

# Cell 7. OpenCV 보조지표 계산 함수

In [8]:
def compute_opencv_stats(image_paths):
    """
    OpenCV 기반 보조지표 계산
    - avg_brightness
    - avg_blue
    - avg_green
    - avg_red
    
    팀원 기준에 맞춰 50x50으로 축소 후 계산
    """
    import cv2

    brightness_values = []
    red_values = []
    green_values = []
    blue_values = []

    for path in image_paths:
        img = cv2.imread(str(path))

        if img is None:
            continue

        # OpenCV 정량 계산용 50x50 축소
        img = cv2.resize(img, (50, 50))

        # OpenCV는 BGR 순서
        b, g, r = cv2.split(img)

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        brightness_values.append(float(np.mean(gray)))
        blue_values.append(float(np.mean(b)))
        green_values.append(float(np.mean(g)))
        red_values.append(float(np.mean(r)))

    if len(brightness_values) == 0:
        return {
            "avg_brightness": None,
            "avg_blue": None,
            "avg_green": None,
            "avg_red": None
        }

    return {
        "avg_brightness": round(float(np.mean(brightness_values)), 3),
        "avg_blue": round(float(np.mean(blue_values)), 3),
        "avg_green": round(float(np.mean(green_values)), 3),
        "avg_red": round(float(np.mean(red_values)), 3)
    }

# Cell 8. JSON 추출 및 파싱 함수

In [9]:
def extract_json_from_output(output_text):
    """
    모델 출력에서 JSON 부분만 추출
    1. ```json ... ``` 형태
    2. { ... } 형태
    둘 다 대응
    """
    markdown_match = re.search(
        r"```json\s*(\{.*?\})\s*```",
        output_text,
        re.DOTALL
    )

    if markdown_match:
        return markdown_match.group(1)

    json_match = re.search(
        r"\{.*\}",
        output_text,
        re.DOTALL
    )

    if json_match:
        return json_match.group(0)

    return output_text


def parse_json_safely(json_str):
    """
    JSON 파싱 안정화
    """
    try:
        return json.loads(json_str)
    except Exception:
        pass

    try:
        return ast.literal_eval(json_str)
    except Exception:
        pass

    fixed = json_str.replace("'", '"')

    try:
        return json.loads(fixed)
    except Exception as e:
        raise ValueError(f"JSON 파싱 실패: {e}")
    
REQUIRED_KEYS = [
    "production_quality",
    "lighting_style",
    "color_mood",
    "editing_pace",
    "motion_graphic",
    "video_format",
    "first_3sec",
    "background_style",
    "top_colors",
    "person_ratio",
    "face_ratio",
    "text_ratio",
    "reason"
]

DEFAULT_VALUES = {
    "production_quality": "미분류",
    "lighting_style": "미분류",
    "color_mood": "미분류",
    "editing_pace": "미분류",
    "motion_graphic": "미분류",
    "video_format": "기타",
    "first_3sec": "장면",
    "background_style": "혼합",
    "top_colors": [],
    "person_ratio": None,
    "face_ratio": None,
    "text_ratio": None,
    "reason": "영상의 주요 시각 요소를 기준으로 판단했습니다."
}

def validate_and_fill_result(result):
    """
    Qwen 출력 결과에서 필수 key 누락 여부를 확인하고,
    누락된 key는 기본값으로 채운다.
    """
    missing_keys = []

    for key in REQUIRED_KEYS:
        if key not in result:
            result[key] = DEFAULT_VALUES[key]
            missing_keys.append(key)

    result["missing_keys"] = missing_keys
    result["is_schema_complete"] = len(missing_keys) == 0

    return result


# Cell 9. 프롬프트 정의

In [10]:
PROMPT_TEXT = """
너는 기업 유튜브 쇼츠 영상을 분석하는 비주얼 마케팅 전문가다.
주어진 이미지들은 하나의 쇼츠 영상에서 1초 단위로 추출된 프레임이다.

반드시 순수 JSON 객체 1개만 출력한다.
마크다운 코드블록, 설명문, 주석은 출력하지 않는다.
모든 값은 반드시 한국어로 작성한다.
특히 top_colors의 경우 한국어 색상명 3개를 리스트로 출력한다.
출력 JSON에는 아래 key만 포함하고, key 이름을 절대 바꾸지 않는다.
reason key는 반드시 포함한다.

[필수 출력 key]
production_quality
lighting_style
color_mood
editing_pace
motion_graphic
video_format
first_3sec
background_style
top_colors
person_ratio
face_ratio
text_ratio
reason

[선택지]
production_quality: ["저예산", "일반", "고퀄리티", "프로페셔널"]
- 저예산: 스마트폰 촬영 느낌, 조명/편집 거의 없음
- 일반: 기본 장비와 간단한 편집
- 고퀄리티: 전문 장비, 컬러그레이딩, 음향, 안정적 촬영
- 프로페셔널: 광고 제작사 수준의 매우 높은 완성도

lighting_style: ["자연광", "인공조명", "역광", "저조도", "혼합"]
- 자연광: 햇빛 중심
- 인공조명: 조명 장비로 통제된 환경
- 역광: 빛이 피사체 뒤에서 들어옴
- 저조도: 전반적으로 어두움
- 혼합: 여러 조명 방식이 섞임

color_mood: ["따뜻함", "차가움", "중립", "비비드", "무채색"]
- 따뜻함: 붉은/노란/주황 계열
- 차가움: 파란/회색/청록 계열
- 중립: 특정 색감으로 치우치지 않음
- 비비드: 채도가 높고 선명함
- 무채색: 흑백 또는 낮은 채도

editing_pace: ["매우 느림", "느림", "보통", "빠름", "매우 빠름"]
- 매우 느림: 컷 전환이 거의 없음
- 느림: 컷 전환이 적고 여유로움
- 보통: 일반적인 편집 속도
- 빠름: 컷 전환이 잦고 역동적
- 매우 빠름: 컷 전환이 매우 잦고 강렬함

motion_graphic: ["없음", "보조적", "핵심요소"]
- 없음: 모션그래픽 거의 없음
- 보조적: 자막/텍스트 애니메이션/간단한 그래픽 사용
- 핵심요소: 모션그래픽이 주된 표현 수단

video_format: ["웹드라마", "브이로그", "시설소개", "에피소드소개", "제품리뷰", "튜토리얼", "광고/CF", "다큐멘터리", "웹예능", "이벤트/행사", "인터뷰", "애니메이션", "기술설명", "요리/레시피", "영양정보", "고객후기", "기타"]
- 웹드라마: 드라마 형식의 브랜디드 콘텐츠
- 브이로그: 일상/현장을 자연스럽게 담은 영상
- 시설소개: 사옥, 매장, 데이터센터 등 공간 소개
- 에피소드소개: 캐릭터/스토리텔링 중심 영상
- 제품리뷰: 제품 사용 경험과 특징 소개
- 튜토리얼: 따라할 수 있는 단계별 안내
- 광고/CF: 짧고 강한 홍보 영상
- 다큐멘터리: 특정 주제를 깊이 다룸
- 웹예능: 게임, 미션, 토크쇼, 리액션 등 오락 구성
- 이벤트/행사: 런칭, 팝업, 기자간담회 등 행사 현장
- 인터뷰: 인물이 직접 말하거나 대화
- 애니메이션: 실사 없이 2D/3D/모션그래픽 중심
- 기술설명: IT 기술 개념이나 구조 설명
- 요리/레시피: 요리 과정이나 식재료 조합
- 영양정보: 영양/건강 정보
- 고객후기: 고객 경험이나 사용 후기
- 기타: 위 분류에 명확히 해당하지 않음

first_3sec: ["텍스트", "인물", "제품", "장면", "기업 로고", "음식"]
- 첫 3초의 주된 요소 1개를 고른다.

background_style: ["실내", "실외", "스튜디오", "혼합"]
- 실내: 건물 내부
- 실외: 야외 공간
- 스튜디오: 통제된 촬영 공간
- 혼합: 여러 배경이 섞임

[비율 기준]
person_ratio: 전체 프레임 중 인물이 등장하는 프레임 비율
face_ratio: 전체 프레임 중 얼굴이 명확히 보이는 프레임 비율
text_ratio: 전체 프레임 중 텍스트/자막이 보이는 프레임 비율
비율은 0.00~1.00 사이 소수 둘째 자리까지 숫자로 출력한다.

[추가 규칙]
- 범주형 항목은 반드시 선택지 중 하나만 사용한다.
- top_colors는 주요 색상 3개를 리스트로 출력한다.
- top_colors는 반드시 한국어 색상명만 사용한다.
- 중국어, 영어 색상명은 '절대' 사용하지 않는다.
- reason은 주요 분류 결과를 뒷받침하는 시각적 근거를 1문장으로 작성한다.
- reason에는 video_format, first_3sec, background_style, person_ratio 판단 근거 중 최소 2개를 반영한다.
- reason은 10자 이상 1문장으로 짧고 자연스럽게 작성한다.
- reason을 확실하게 정하기 어려우면 "영상의 주요 시각 요소를 기준으로 판단했습니다."라고 작성한다.
- 어려운 표현, 번역투, 이상한 단어를 쓰지 않는다.
- 확실하지 않으면 가장 가까운 범주를 선택한다.
- video_format 판단은 등장 소재보다 영상 목적을 우선한다.
- video_format 판단 시 영상에 음식이 등장하더라도, 조리 과정을 단계별로 설명하지 않고 제품/브랜드 홍보 목적이 강하면 "광고/CF"로 분류한다.
- "요리/레시피"는 실제 조리 과정이나 레시피 안내가 중심일 때만 선택한다.
"""

# Cell 10. 영상 1개 Qwen 분석 함수

In [11]:
def infer_one(
    model,
    processor,
    video_id,
    image_paths,
    max_new_tokens=512,
    max_retries=3
):
    """
    하나의 video_id에 대해 Qwen VLM 분석 수행
    image_paths는 1초당 1프레임 기준으로 이미 추출된 전체 프레임 목록
    """
    if len(image_paths) == 0:
        raise ValueError(f"{video_id}: 입력 이미지가 없습니다.")

    images = load_images(image_paths, image_size=IMAGE_SIZE)

    messages = [
        {
            "role": "user",
            "content": [
                *[{"type": "image", "image": img} for img in images],
                {"type": "text", "text": PROMPT_TEXT}
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=images,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    last_output = ""

    for attempt in range(max_retries):
        torch.cuda.empty_cache()

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.1
            )

        generated_ids_trimmed = [
            out_ids[len(in_ids):]
            for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0]

        last_output = output_text

        try:
            json_str = extract_json_from_output(output_text)
            result = parse_json_safely(json_str)

            # 필수 key 검증 및 누락값 보정
            result = validate_and_fill_result(result)

            result["video_id"] = video_id
            result["model_name"] = "Qwen2.5-VL-3B-Instruct-4bit"
            result["frame_sampling_method"] = "1fps_all_frames"
            result["used_frame_count"] = len(image_paths)

            return result

        except Exception as e:
            print(f"[{video_id}] JSON 파싱 실패 {attempt + 1}/{max_retries}")
            print("출력 일부:", output_text[:300])

    raise RuntimeError(
        f"{video_id} 분석 실패. 마지막 출력:\n{last_output}"
    )

# Cell 11. 영상 1개 dry run

In [12]:
video_id = video_dirs[0].name

image_paths = gather_images(FRAMES_ROOT / video_id)

# 1초당 1프레임 기준으로 이미 저장되어 있으므로 전체 사용
used_paths = image_paths

print("video_id:", video_id)
print("사용 프레임 수:", len(used_paths))

for p in used_paths[:5]:
    print(p)

video_id: knCQBlBKSRM
사용 프레임 수: 21
data\frames_for_vlm\knCQBlBKSRM\frame_000.jpg
data\frames_for_vlm\knCQBlBKSRM\frame_001.jpg
data\frames_for_vlm\knCQBlBKSRM\frame_002.jpg
data\frames_for_vlm\knCQBlBKSRM\frame_003.jpg
data\frames_for_vlm\knCQBlBKSRM\frame_004.jpg


# Cell 12. 영상 1개 Qwen 분석 실행

In [13]:
vlm_result = infer_one(
    model=model,
    processor=processor,
    video_id=video_id,
    image_paths=used_paths,
    max_new_tokens=MAX_NEW_TOKENS,
    max_retries=3
)

print(json.dumps(vlm_result, ensure_ascii=False, indent=2))

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{
  "production_quality": "고퀄리티",
  "lighting_style": "인공조명",
  "color_mood": "차가움",
  "editing_pace": "빠름",
  "motion_graphic": "보조적",
  "video_format": "광고/CF",
  "first_3sec": "텍스트",
  "background_style": "실외",
  "top_colors": [
    "파랑",
    "검정",
    "갈색"
  ],
  "person_ratio": 0.5,
  "face_ratio": 0.7,
  "text_ratio": 0.9,
  "reason": "첫 3초의 프레임에서는 텍스트가 나타나며, 인물이 등장하는 부분은 대부분 얼굴만 보여주고, 배경은 야외로 설정되어 있어 광고/CF 형식의 영상임을 알 수 있습니다.",
  "missing_keys": [],
  "is_schema_complete": true,
  "video_id": "knCQBlBKSRM",
  "model_name": "Qwen2.5-VL-3B-Instruct-4bit",
  "frame_sampling_method": "1fps_all_frames",
  "used_frame_count": 21
}


# Cell 13. 영상 1개 OpenCV 지표 병합

In [14]:
opencv_stats = compute_opencv_stats(used_paths)

final_result = {
    **vlm_result,
    **opencv_stats,
    "original_frame_count": len(image_paths),
    "used_frame_count": len(used_paths)
}

print(json.dumps(final_result, ensure_ascii=False, indent=2))

{
  "production_quality": "고퀄리티",
  "lighting_style": "인공조명",
  "color_mood": "차가움",
  "editing_pace": "빠름",
  "motion_graphic": "보조적",
  "video_format": "광고/CF",
  "first_3sec": "텍스트",
  "background_style": "실외",
  "top_colors": [
    "파랑",
    "검정",
    "갈색"
  ],
  "person_ratio": 0.5,
  "face_ratio": 0.7,
  "text_ratio": 0.9,
  "reason": "첫 3초의 프레임에서는 텍스트가 나타나며, 인물이 등장하는 부분은 대부분 얼굴만 보여주고, 배경은 야외로 설정되어 있어 광고/CF 형식의 영상임을 알 수 있습니다.",
  "missing_keys": [],
  "is_schema_complete": true,
  "video_id": "knCQBlBKSRM",
  "model_name": "Qwen2.5-VL-3B-Instruct-4bit",
  "frame_sampling_method": "1fps_all_frames",
  "used_frame_count": 21,
  "avg_brightness": 34.737,
  "avg_blue": 28.688,
  "avg_green": 34.006,
  "avg_red": 38.514,
  "original_frame_count": 21
}


# Cell 14. 전체 영상 자동 분석

In [15]:
all_results = []

print("Qwen VLM 전체 영상 분석 시작")

for idx, video_dir in enumerate(video_dirs):
    video_id = video_dir.name

    print(f"\n[{idx + 1}/{len(video_dirs)}] 분석 중: {video_id}")

    try:
        image_paths = gather_images(video_dir)

        # 1초당 1프레임 기준 전체 사용
        used_paths = image_paths

        print(f"사용 프레임 수: {len(used_paths)}")

        vlm_result = infer_one(
            model=model,
            processor=processor,
            video_id=video_id,
            image_paths=used_paths,
            max_new_tokens=MAX_NEW_TOKENS,
            max_retries=3
        )

        opencv_stats = compute_opencv_stats(used_paths)

        final_result = {
            **vlm_result,
            **opencv_stats,
            "original_frame_count": len(image_paths),
            "used_frame_count": len(used_paths),
            "frame_sampling_method": "1fps_all_frames"
        }

        # Qwen 출력에서 필수 key 누락 여부
        # final_result.pop("missing_keys", None)
        # final_result.pop("is_schema_complete", None)

        all_results.append(final_result)

        df_result = pd.DataFrame([final_result])

        if not CSV_PATH.exists():
            df_result.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
        else:
            df_result.to_csv(
                CSV_PATH,
                mode="a",
                header=False,
                index=False,
                encoding="utf-8-sig"
            )

        print("분석 완료 및 CSV 저장")

    except Exception as e:
        print(f"분석 실패: {video_id}")
        print(e)
        continue

with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("\n전체 작업 완료")
print("CSV 저장 위치:", CSV_PATH)
print("JSON 저장 위치:", JSON_PATH)


Qwen VLM 전체 영상 분석 시작

[1/8] 분석 중: knCQBlBKSRM
사용 프레임 수: 21
분석 완료 및 CSV 저장

[2/8] 분석 중: LRmLOsmMHts
사용 프레임 수: 30
분석 완료 및 CSV 저장

[3/8] 분석 중: mfi0n3oNABM
사용 프레임 수: 56
분석 완료 및 CSV 저장

[4/8] 분석 중: MIJR3Dm0YOk
사용 프레임 수: 50
분석 완료 및 CSV 저장

[5/8] 분석 중: NCVxpXxTMSU
사용 프레임 수: 12
분석 완료 및 CSV 저장

[6/8] 분석 중: twi9zUxsXu0
사용 프레임 수: 55
분석 완료 및 CSV 저장

[7/8] 분석 중: Wkiz8JVPvXA
사용 프레임 수: 30
분석 완료 및 CSV 저장

[8/8] 분석 중: Xfu1kCID0Ls
사용 프레임 수: 23
분석 완료 및 CSV 저장

전체 작업 완료
CSV 저장 위치: results\qwen_vlm_1fps_results.csv
JSON 저장 위치: results\qwen_vlm_1fps_results.json


# Cell 15. 결과 확인

In [16]:
df = pd.read_csv(CSV_PATH)

print("결과 행 수:", len(df))
df.head()

결과 행 수: 8


,production_quality,lighting_style,color_mood,editing_pace,motion_graphic,video_format,first_3sec,background_style,top_colors,person_ratio,...,is_schema_complete,video_id,model_name,frame_sampling_method,used_frame_count,avg_brightness,avg_blue,avg_green,avg_red,original_frame_count
0,고퀄리티,인공조명,차가움,빠름,보조적,광고/CF,텍스트,실외,"['파랑', '검정', '갈색']",0.50,...,True,knCQBlBKSRM,Qwen2.5-VL-3B-Instruct-4bit,1fps_all_frames,21,34.737,28.688,34.006,38.514,21
1,고퀄리티,인공조명,따뜻함,빠름,보조적,광고/CF,인물,실내,"['파랑', '노랑', '빨강']",0.70,...,True,LRmLOsmMHts,Qwen2.5-VL-3B-Instruct-4bit,1fps_all_frames,30,195.093,196.727,193.490,197.750,30
2,고퀄리티,자연광,따뜻함,빠름,보조적,광고/CF,인물,실외,"['핑크', '초록', '갈색']",0.60,...,True,mfi0n3oNABM,Qwen2.5-VL-3B-Instruct-4bit,1fps_all_frames,56,115.403,107.136,120.163,109.228,56
3,고퀄리티,인공조명,차가움,빠름,보조적,웹예능,인물,스튜디오,"['검정', '화이트', '파랑']",0.75,...,True,MIJR3Dm0YOk,Qwen2.5-VL-3B-Instruct-4bit,1fps_all_frames,50,26.621,25.146,25.716,28.958,50
4,고퀄리티,인공조명,차가움,빠름,보조적,광고/CF,텍스트,실내,"['검정', '파랑', '갈색']",0.50,...,True,NCVxpXxTMSU,Qwen2.5-VL-3B-Instruct-4bit,1fps_all_frames,12,59.565,46.589,58.174,67.237,12


In [18]:
check_cols = [
    "video_id",
    "model_name",
    "video_format",
    "production_quality",
    "lighting_style",
    "color_mood",
    "editing_pace",
    "motion_graphic",
    "person_ratio",
    "face_ratio",
    "text_ratio",
    "avg_brightness",
    "avg_red",
    "avg_green",
    "avg_blue",
    "used_frame_count"
]

df[check_cols]

,video_id,model_name,video_format,production_quality,lighting_style,color_mood,editing_pace,motion_graphic,person_ratio,face_ratio,text_ratio,avg_brightness,avg_red,avg_green,avg_blue,used_frame_count
0,mfi0n3oNABM,Qwen2.5-VL-3B-Instruct-4bit,브이로그,고퀄리티,자연광,따뜻함,느림,보조적,0.5,0.70,0.1,115.403,109.228,120.163,107.136,56
1,Wkiz8JVPvXA,Qwen2.5-VL-3B-Instruct-4bit,요리/레시피,고퀄리티,인공조명,따뜻함,빠름,보조적,0.5,0.80,0.2,115.007,134.701,112.499,76.296,30
2,mfi0n3oNABM,Qwen2.5-VL-3B-Instruct-4bit,브이로그,고퀄리티,자연광,따뜻함,느림,보조적,0.6,0.85,0.4,115.403,109.228,120.163,107.136,56
3,Wkiz8JVPvXA,Qwen2.5-VL-3B-Instruct-4bit,광고/CF,고퀄리티,인공조명,따뜻함,빠름,보조적,0.5,0.80,0.2,115.007,134.701,112.499,76.296,30
4,mfi0n3oNABM,Qwen2.5-VL-3B-Instruct-4bit,광고/CF,고퀄리티,자연광,따뜻함,빠름,보조적,0.5,0.70,0.4,115.403,109.228,120.163,107.136,56
5,Wkiz8JVPvXA,Qwen2.5-VL-3B-Instruct-4bit,광고/CF,고퀄리티,인공조명,따뜻함,빠름,보조적,0.5,0.80,0.4,115.007,134.701,112.499,76.296,30


# Cell 16. GPU 메모리 정리

In [19]:
torch.cuda.empty_cache()
print("CUDA cache 정리 완료")

CUDA cache 정리 완료
